<a href="https://colab.research.google.com/github/ayesha0859/DL/blob/main/transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#!pip install transformers datasets torch -q

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

In [ ]:
dataset = load_dataset("ag_news")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

In [ ]:
def tokenize(example):
    return tokenizer(example['text'], padding='max_length', truncation=True)

In [ ]:
dataset = dataset.map(tokenize, batched=True)

In [ ]:
dataset = dataset.rename_column("label", "labels")

In [ ]:
dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=4)

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=1,
    logging_dir="./logs",
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"].select(range(2000)),
    eval_dataset=dataset["test"].select(range(500))
)

In [ ]:
trainer.train()

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)

text = "India won the T20 series"

inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)

# 🔥 Move inputs to device
inputs = {key: value.to(device) for key, value in inputs.items()}

outputs = model(**inputs)
predicted_class = torch.argmax(outputs.logits).item()

labels = ["World", "Sports", "Business", "Sci/Tech"]

print("Text:", text)
print("Predicted Category:", labels[predicted_class])